<a href="https://colab.research.google.com/github/OJB-Quantum/Monte-Carlo-Sim/blob/main/Monte_Carlo_Sim_of_a_200_Year_Lifespan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random
import statistics

def mortality_probability(age, max_age=200):
    """
    A toy function that returns the probability of death at a given age.
    The function is designed so that average lifespan hovers near 200,
    but you can adjust it as needed.
    """
    # Example: A small base rate, increasing with age.
    # At age 0 --> prob ~ 0.0, at age 200 --> prob ~ 0.02
    # This will lead to many individuals reaching 200,
    # but is just a toy model. Tweak as necessary.
    return 0.0001 * age

def simulate_one_life(max_age=200):
    """
    Simulate a single life path, returning:
      - age of death (or 200 if still alive)
      - record of 'career changes'
      - record of 'major illnesses'
    """
    current_age = 0
    is_alive = True

    career_change_years = []
    illness_years = []

    while is_alive and current_age <= max_age:
        # Roll for mortality this year
        death_chance = mortality_probability(current_age, max_age)
        if random.random() < death_chance:
            # Person dies this year
            return current_age, career_change_years, illness_years

        # If still alive, check for events:
        # 1) Career change (example ages 25-150, 5% chance each year)
        if 25 <= current_age <= 150:
            if random.random() < 0.05:
                career_change_years.append(current_age)

        # 2) Major illness (example ages 40-190, 2% chance each year)
        if 40 <= current_age <= 190:
            if random.random() < 0.02:
                illness_years.append(current_age)

        current_age += 1

    # If we exited the loop without dying, it means we reached max_age
    return max_age, career_change_years, illness_years


def run_simulation(num_simulations=10000, max_age=200):
    """
    Run multiple simulations and return:
     - distribution of death ages
     - average age of death
     - average number of career changes
     - average number of major illnesses
    """
    death_ages = []
    career_change_counts = []
    illness_counts = []

    for _ in range(num_simulations):
        death_age, career_changes, illnesses = simulate_one_life(max_age)
        death_ages.append(death_age)
        career_change_counts.append(len(career_changes))
        illness_counts.append(len(illnesses))

    avg_death_age = statistics.mean(death_ages)
    avg_career_changes = statistics.mean(career_change_counts)
    avg_illnesses = statistics.mean(illness_counts)

    results = {
        "avg_death_age": avg_death_age,
        "avg_career_changes": avg_career_changes,
        "avg_illnesses": avg_illnesses,
        "death_age_distribution": death_ages
    }
    return results

# -----------------------
# Run the simulation
# -----------------------
if __name__ == "__main__":
    np.random.seed(42)  # For reproducibility (optional)
    random.seed(42)

    N = 10_000
    simulation_results = run_simulation(num_simulations=N, max_age=200)

    print(f"Ran {N} simulations of a 200-year lifespan model.\n")
    print(f"Average age of death: {simulation_results['avg_death_age']:.2f}")
    print(f"Average number of career changes: {simulation_results['avg_career_changes']:.2f}")
    print(f"Average number of major illnesses: {simulation_results['avg_illnesses']:.2f}")
    # You could also plot or further analyze death_age_distribution if desired.

Ran 10000 simulations of a 200-year lifespan model.

Average age of death: 120.22
Average number of career changes: 4.22
Average number of major illnesses: 1.61


In [2]:
# Monte Carlo Simulation: CPU vs GPU (CuPy) Comparison
# ----------------------------------------------------
# This code demonstrates how to run the same Monte Carlo simulation
# on both the CPU (NumPy + Python) and the GPU (CuPy) in Google Colab.
# It computes:
#  - Age of death distribution
#  - Average number of career changes
#  - Average number of major illnesses
#    for N simulated lives (max_age = 200).
# We measure runtimes for both implementations.

import time
import numpy as np
import random
import statistics

# -----------------------
# 1) CPU Implementation
# -----------------------

def mortality_probability(age, max_age=200):
    """
    Toy function returning probability of death at a given age.
    """
    return 0.0001 * age


def simulate_one_life_cpu(max_age=200):
    """
    Simulate one life path on CPU.
    Returns: (death_age, career_changes_count, illness_count)
    """
    current_age = 0
    career_changes = 0
    illnesses = 0
    while current_age <= max_age:
        # Mortality check
        if random.random() < mortality_probability(current_age, max_age):
            return current_age, career_changes, illnesses
        # Career change (ages 25-150, 5% per year)
        if 25 <= current_age <= 150:
            if random.random() < 0.05:
                career_changes += 1
        # Major illness (ages 40-190, 2% per year)
        if 40 <= current_age <= 190:
            if random.random() < 0.02:
                illnesses += 1
        current_age += 1
    # Survived full lifespan
    return max_age, career_changes, illnesses


def run_simulation_cpu(num_simulations=10000, max_age=200):
    """
    Run multiple CPU simulations. Return results dict.
    """
    death_ages = []
    career_counts = []
    illness_counts = []
    for _ in range(num_simulations):
        d_age, careers, ill = simulate_one_life_cpu(max_age)
        death_ages.append(d_age)
        career_counts.append(careers)
        illness_counts.append(ill)
    avg_death_age = statistics.mean(death_ages)
    avg_careers = statistics.mean(career_counts)
    avg_illnesses = statistics.mean(illness_counts)
    return {
        "avg_death_age": avg_death_age,
        "avg_career_changes": avg_careers,
        "avg_illnesses": avg_illnesses,
        "death_age_distribution": np.array(death_ages)
    }

# -----------------------
# 2) GPU Implementation
# -----------------------
# We use CuPy to vectorize the simulation across all lives simultaneously.
# CuPy is a drop-in replacement for NumPy that runs on CUDA GPUs.

try:
    import cupy as cp
except ImportError:
    # In Colab, install CuPy if not already installed
    !pip install cupy-cuda11x --quiet
    import cupy as cp


def run_simulation_gpu(num_simulations=10000, max_age=200):
    """
    Run vectorized GPU simulation using CuPy.
    Returns results dict (with arrays on CPU for easy comparison).
    """
    # Seed CuPy's RNG (this may differ slightly from Python/NumPy seeds)
    cp.random.seed(42)

    # 1) Prepare age grid [0, 1, 2, ..., max_age]
    ages = cp.arange(max_age + 1, dtype=cp.int32)  # shape: (max_age+1,)
    # Compute mortality probabilities for each age
    probs = 0.0001 * ages.astype(cp.float32)  # shape: (max_age+1,)
    probs = probs[:, cp.newaxis]  # shape: (max_age+1, 1)

    # 2) Generate random numbers for each (age, life) pair
    #    Shape: (max_age+1, num_simulations)
    rand_matrix = cp.random.random((max_age + 1, num_simulations), dtype=cp.float32)

    # 3) Create a boolean mask of where death occurs (rand < prob)
    death_mask = rand_matrix < probs  # shape: (max_age+1, num_simulations)

    # 4) Find death ages: first index along axis=0 where death occurs
    #    For columns with no True, we set death_age = max_age
    has_death = death_mask.any(axis=0)  # shape: (num_simulations,)
    # argmax returns 0 if all False, so we adjust later
    first_idx = death_mask.argmax(axis=0)  # shape: (num_simulations,)
    death_ages_gpu = cp.where(has_death, first_idx, max_age).astype(cp.int32)

    # 5) Career changes: ages 25-150, 5% per year
    career_age_start, career_age_end = 25, 150
    num_career_years = career_age_end - career_age_start + 1
    career_probs = 0.05 * cp.ones((num_career_years, 1), dtype=cp.float32)
    career_rand = cp.random.random((num_career_years, num_simulations), dtype=cp.float32)
    career_occurrences = career_rand < career_probs  # boolean mask
    career_counts_gpu = career_occurrences.sum(axis=0)  # shape: (num_simulations,)

    # 6) Major illnesses: ages 40-190, 2% per year
    ill_age_start, ill_age_end = 40, 190
    num_ill_years = ill_age_end - ill_age_start + 1
    ill_probs = 0.02 * cp.ones((num_ill_years, 1), dtype=cp.float32)
    ill_rand = cp.random.random((num_ill_years, num_simulations), dtype=cp.float32)
    ill_occurrences = ill_rand < ill_probs
    ill_counts_gpu = ill_occurrences.sum(axis=0)

    # 7) Transfer results back to CPU (NumPy) arrays
    death_ages = cp.asnumpy(death_ages_gpu)
    career_counts = cp.asnumpy(career_counts_gpu)
    ill_counts = cp.asnumpy(ill_counts_gpu)

    avg_death_age = death_ages.mean()
    avg_careers = career_counts.mean()
    avg_illnesses = ill_counts.mean()

    return {
        "avg_death_age": float(avg_death_age),
        "avg_career_changes": float(avg_careers),
        "avg_illnesses": float(avg_illnesses),
        "death_age_distribution": death_ages
    }

# -----------------------
# 3) Run & Compare Timings
# -----------------------

if __name__ == "__main__":
    N = 50_000  # Increase N for more pronounced GPU speedup
    max_age = 200

    # Seed CPU RNGs for reproducibility
    np.random.seed(42)
    random.seed(42)

    # 3a) CPU timing
    start_cpu = time.time()
    cpu_results = run_simulation_cpu(num_simulations=N, max_age=max_age)
    end_cpu = time.time()
    cpu_time = end_cpu - start_cpu

    print("CPU Results:")
    print(f"  - Avg age of death: {cpu_results['avg_death_age']:.2f}")
    print(f"  - Avg career changes: {cpu_results['avg_career_changes']:.2f}")
    print(f"  - Avg illnesses: {cpu_results['avg_illnesses']:.2f}")
    print(f"  - CPU runtime: {cpu_time:.3f} seconds\n")

    # 3b) GPU timing (with CuPy kernels executed synchronously)
    # Warm-up: run once to initialize CUDA context
    _ = run_simulation_gpu(num_simulations=1_000, max_age=max_age)
    cp.cuda.Stream.null.synchronize()

    start_gpu = time.time()
    gpu_results = run_simulation_gpu(num_simulations=N, max_age=max_age)
    cp.cuda.Stream.null.synchronize()
    end_gpu = time.time()
    gpu_time = end_gpu - start_gpu

    print("GPU (CuPy) Results:")
    print(f"  - Avg age of death: {gpu_results['avg_death_age']:.2f}")
    print(f"  - Avg career changes: {gpu_results['avg_career_changes']:.2f}")
    print(f"  - Avg illnesses: {gpu_results['avg_illnesses']:.2f}")
    print(f"  - GPU runtime: {gpu_time:.3f} seconds\n")

    # 3c) Speedup ratio
    speedup = cpu_time / gpu_time if gpu_time > 0 else float('inf')
    print(f"Speedup (CPU_time / GPU_time): {speedup:.2f}x")

CPU Results:
  - Avg age of death: 119.95
  - Avg career changes: 4.22
  - Avg illnesses: 1.60
  - CPU runtime: 2.170 seconds

GPU (CuPy) Results:
  - Avg age of death: 119.72
  - Avg career changes: 6.30
  - Avg illnesses: 3.02
  - GPU runtime: 0.005 seconds

Speedup (CPU_time / GPU_time): 450.75x


In [3]:
# Monte Carlo Sankey Pipeline: CPU vs GPU (CuPy) Comparison
# --------------------------------------------------------
# This code demonstrates how to:
#  1) Run the same Monte Carlo life simulation on CPU and GPU
#  2) Build Sankey flow counts (career->illness, illness->death_age)
#  3) Plot a Sankey diagram via Plotly
#  4) Compare runtimes for CPU vs GPU implementations

import time
import numpy as np
import random
import statistics
import plotly.graph_objects as go
from collections import Counter

# ----------------------
# 1) Common Definitions
# ----------------------

def mortality_probability(age, max_age=200):
    return 0.0001 * age

# Binning functions (used by both CPU and GPU paths)
def bin_career_changes(count):
    if count == 0:
        return "Career=0"
    elif count == 1:
        return "Career=1"
    elif count == 2:
        return "Career=2"
    else:
        return "Career=3+"

def bin_illnesses(count):
    if count == 0:
        return "Ill=0"
    elif count == 1:
        return "Ill=1"
    elif count == 2:
        return "Ill=2"
    else:
        return "Ill=3+"

def bin_death_age(age):
    if age <= 50:
        return "Age=0-50"
    elif age <= 100:
        return "Age=51-100"
    elif age <= 150:
        return "Age=101-150"
    else:
        return "Age=151-200"

# Build Sankey data from flow counters
def build_sankey_data(flow1_counter, flow2_counter):
    career_bins = ["Career=0", "Career=1", "Career=2", "Career=3+"]
    illness_bins = ["Ill=0", "Ill=1", "Ill=2", "Ill=3+"]
    age_bins = ["Age=0-50", "Age=51-100", "Age=101-150", "Age=151-200"]

    all_nodes = career_bins + illness_bins + age_bins
    label_to_index = {label: i for i, label in enumerate(all_nodes)}

    source_list = []
    target_list = []
    value_list = []

    # (career -> illness)
    for (c_bin, i_bin), val in flow1_counter.items():
        source_list.append(label_to_index[c_bin])
        target_list.append(label_to_index[i_bin])
        value_list.append(val)

    # (illness -> age)
    for (i_bin, a_bin), val in flow2_counter.items():
        source_list.append(label_to_index[i_bin])
        target_list.append(label_to_index[a_bin])
        value_list.append(val)

    return all_nodes, source_list, target_list, value_list

# Plot Sankey via Plotly
def plot_sankey(all_nodes, source_list, target_list, value_list, title="Life Simulation Sankey"):
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_nodes
        ),
        link=dict(
            source=source_list,
            target=target_list,
            value=value_list
        )
    )])
    fig.update_layout(title_text=title, font_size=12)
    fig.show()

# ----------------------
# 2) CPU Implementation
# ----------------------

def simulate_one_life_cpu(max_age=200):
    current_age = 0
    career_change_years = []
    illness_years = []
    while current_age <= max_age:
        if random.random() < mortality_probability(current_age, max_age):
            return current_age, len(career_change_years), len(illness_years)
        if 25 <= current_age <= 150 and random.random() < 0.05:
            career_change_years.append(current_age)
        if 40 <= current_age <= 190 and random.random() < 0.02:
            illness_years.append(current_age)
        current_age += 1
    return max_age, len(career_change_years), len(illness_years)


def run_simulation_cpu(num_simulations=10000, max_age=200):
    death_ages = []
    career_counts = []
    illness_counts = []
    for _ in range(num_simulations):
        d_age, careers, ills = simulate_one_life_cpu(max_age)
        death_ages.append(d_age)
        career_counts.append(careers)
        illness_counts.append(ills)
    return {
        "death_ages": np.array(death_ages, dtype=np.int32),
        "career_counts": np.array(career_counts, dtype=np.int32),
        "illness_counts": np.array(illness_counts, dtype=np.int32)
    }

# Build flow counters (career->illness, illness->death_age)
def build_flow_counts_cpu(death_ages, career_counts, illness_counts):
    combos = []
    for c_count, i_count, d_age in zip(career_counts, illness_counts, death_ages):
        c_bin = bin_career_changes(int(c_count))
        i_bin = bin_illnesses(int(i_count))
        a_bin = bin_death_age(int(d_age))
        combos.append((c_bin, i_bin, a_bin))
    combo_counter = Counter(combos)
    flow1_counter = Counter()
    flow2_counter = Counter()
    for (c_bin, i_bin, a_bin), cnt in combo_counter.items():
        flow1_counter[(c_bin, i_bin)] += cnt
        flow2_counter[(i_bin, a_bin)] += cnt
    return flow1_counter, flow2_counter

# ----------------------
# 3) GPU Implementation (CuPy)
# ----------------------
try:
    import cupy as cp
except ImportError:
    # Install matching CuPy for Colab's CUDA version (e.g., CUDA 11.6)
    !pip install --quiet cupy-cuda116
    import cupy as cp


def run_simulation_gpu(num_simulations=10000, max_age=200):
    cp.random.seed(42)
    # Age grid
    ages = cp.arange(max_age + 1, dtype=cp.int32)  # (max_age+1,)
    probs = (0.0001 * ages.astype(cp.float32))[:, cp.newaxis]  # (max_age+1,1)

    # Mortality randoms: shape (max_age+1, num_simulations)
    rand_mort = cp.random.random((max_age + 1, num_simulations), dtype=cp.float32)
    death_mask = rand_mort < probs  # boolean mask
    has_death = death_mask.any(axis=0)
    first_idx = death_mask.argmax(axis=0)
    death_ages_gpu = cp.where(has_death, first_idx, max_age).astype(cp.int32)

    # Career changes (ages 25–150)
    c_start, c_end = 25, 150
    num_c_years = c_end - c_start + 1
    career_probs = 0.05 * cp.ones((num_c_years, 1), dtype=cp.float32)
    rand_career = cp.random.random((num_c_years, num_simulations), dtype=cp.float32)
    career_counts_gpu = (rand_career < career_probs).sum(axis=0).astype(cp.int32)

    # Illnesses (ages 40–190)
    i_start, i_end = 40, 190
    num_i_years = i_end - i_start + 1
    illness_probs = 0.02 * cp.ones((num_i_years, 1), dtype=cp.float32)
    rand_ill = cp.random.random((num_i_years, num_simulations), dtype=cp.float32)
    illness_counts_gpu = (rand_ill < illness_probs).sum(axis=0).astype(cp.int32)

    # Transfer to CPU (NumPy)
    death_ages = cp.asnumpy(death_ages_gpu)
    career_counts = cp.asnumpy(career_counts_gpu)
    illness_counts = cp.asnumpy(illness_counts_gpu)

    return death_ages, career_counts, illness_counts

# Build flow counters on CPU from GPU-generated arrays (reuse build_flow_counts_cpu)
# ----------------------

if __name__ == "__main__":
    N = 10000  # Number of simulated lives (adjust as needed)
    max_age = 200

    # -------- CPU Path --------
    np.random.seed(42)
    random.seed(42)
    start_cpu = time.time()
    cpu_death, cpu_career, cpu_ill = run_simulation_cpu(num_simulations=N, max_age=max_age).values()
    flow1_cpu, flow2_cpu = build_flow_counts_cpu(cpu_death, cpu_career, cpu_ill)
    all_nodes_cpu, src_cpu, tgt_cpu, val_cpu = build_sankey_data(flow1_cpu, flow2_cpu)
    cpu_time = time.time() - start_cpu

    print("CPU Path Completed")
    print(f"  - CPU runtime (simulation + flow counts): {cpu_time:.3f} seconds")

    # -------- GPU Path --------
    # Warm up CUDA context
    _ = run_simulation_gpu(num_simulations=1000, max_age=max_age)
    cp.cuda.Stream.null.synchronize()
    np.random.seed(42)
    random.seed(42)
    start_gpu = time.time()
    gpu_death, gpu_career, gpu_ill = run_simulation_gpu(num_simulations=N, max_age=max_age)
    flow1_gpu, flow2_gpu = build_flow_counts_cpu(gpu_death, gpu_career, gpu_ill)
    all_nodes_gpu, src_gpu, tgt_gpu, val_gpu = build_sankey_data(flow1_gpu, flow2_gpu)
    cp.cuda.Stream.null.synchronize()
    gpu_time = time.time() - start_gpu

    print("GPU Path Completed")
    print(f"  - GPU runtime (simulation + flow counts): {gpu_time:.3f} seconds")

    speedup = cpu_time / gpu_time if gpu_time > 0 else float('inf')
    print(f"Speedup (CPU_time / GPU_time): {speedup:.2f}x")

    # -------- Sankey Plot (use CPU-generated Sankey data) --------
    # Plot only once, using CPU Sankey data (same topology as GPU)
    plot_sankey(all_nodes_cpu, src_cpu, tgt_cpu, val_cpu,
                title="Life Simulation Sankey (CPU-generated data)")

CPU Path Completed
  - CPU runtime (simulation + flow counts): 0.270 seconds
GPU Path Completed
  - GPU runtime (simulation + flow counts): 0.008 seconds
Speedup (CPU_time / GPU_time): 32.58x
